[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_2_Selecting_Rows.ipynb)

# 07.2: Selecting Rows

Notebook 07.1 selected columns. This notebook selects rows, which is how you narrow a dataset down to the part that answers your question. The main tool is the boolean mask from notebook 06.2. It works the same way here, except that it now keeps whole rows.

## The data

We use the Broadway data again, where each row is one show in one week.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df.shape

## Filtering with a mask

A comparison on a column gives a boolean series, with one value per row. Put that mask inside the square brackets to keep the rows where it is `True`.

In [ ]:
year_2000 = df[df["Year"] == 2000]
print(year_2000.shape)
year_2000.head(3)

There are 1524 rows from the year 2000, and all 12 columns are still there. Since the result is a DataFrame, you can use any column of it. Be careful with `==`, because a single `=` means assignment and would be an error here.

## Selecting rows and columns together: `.loc`

`.loc` from module 06 takes two things on a DataFrame, separated by a comma: which rows, and then which columns. A mask can pick the rows, and a list can pick the columns.

In [ ]:
df.loc[df["Year"] == 2000, ["Name", "Type", "Gross"]].head(3)

You can also look up one row by its index label and one column by name, which gives a single value. Using `:` for the rows means all rows.

In [ ]:
print(df.loc[0, "Name"])
print(df.loc[:, "Gross"].max())

The first line prints Tru, the show in the row labeled 0. The second prints 3201333, the largest weekly gross in the data.

## Combining conditions

The rules are the same as in 06.2: use `&`, `|` and `~`, with parentheses around each condition. Which rows are musicals from the year 2000?

In [ ]:
musicals_2000 = df[(df["Type"] == "Musical") & (df["Year"] == 2000)]
print(len(musicals_2000))
print(musicals_2000["Name"].nunique())

There are 1113 rows but only 37 different musicals. Again, this is because each row is one week, so a musical that ran all year has about 52 rows. It is easy to report 1113 by accident when the question is how many musicals there were, so always ask what one row represents.

`.isin()` is the short way to match several values in one column. Here are the rows for three long-running musicals.

In [ ]:
big_three = df[df["Name"].isin(["The Lion King", "Wicked", "The Phantom Of The Opera"])]
print(len(big_three))
print(big_three["Gross"].sum())
print(df["Gross"].sum())

Those three shows account for 3119822366 of the 18654569593 dollars in the data, which is one way to answer how much of Broadway's income came from its biggest shows. We can get the share by dividing.

In [ ]:
big_three["Gross"].sum() / df["Gross"].sum()

That is about 0.167. Three of 820 shows brought in about 17 percent of all the gross in this data.

## Text and masks

The `.str` methods from notebook 06.4 work on a column, and they give masks that select rows.

In [ ]:
df[df["Name"].str.contains("Phantom")]["Name"].unique()

This returns just one show, The Phantom Of The Opera. Checking with `.unique()` like this is a good way to make sure that a text search found what you meant, and did not also pick up other shows.

## Sorting rows and finding the top N

On a DataFrame, `.sort_values()` needs to know which column to sort by. Combining it with `.head()` gives the top rows, as it did for a series.

In [ ]:
df.sort_values("Gross", ascending=False).head(3)[["Name", "Full", "Gross"]]

Those are the highest-grossing weeks in the data, and the top one is Wicked in the week of December 29, 2013, with 3201333. Each step in this chain returns a DataFrame, so you can keep adding to it. Filtering first lets you ask narrower questions, such as which musical had the largest week in 2000.

In [ ]:
musicals_2000.sort_values("Gross", ascending=False).head(1)[["Name", "Gross"]]

Aida had the largest week among musicals in 2000, at 1155646.

## Changing a filtered result

When you filter a DataFrame and then plan to change the result, such as adding a column, call `.copy()` on it. This makes clear that the result is its own table, and it prevents a warning that pandas may give otherwise. It also guarantees that `df` is not changed.

In [ ]:
plays = df[df["Type"] == "Play"].copy()
plays["Gross_Millions"] = plays["Gross"] / 1000000
plays[["Name", "Gross_Millions"]].head(2)

## What's next

You can now pull out any rows and columns you want. Before you trust an answer, though, you need to know whether the data behind it is sound, and you saw a warning sign in 07.1 with a `Capacity` of 892. Notebook 07.3 shows how to spot problems like this one and what to do about them.